# Chapter 6 — Direct Methods for Solving Linear Systems

**Burden & Faires, *Numerical Analysis*, 10th ed.**

---

This session covers:

| Section | Topic |
|---------|-------|
| 6.1 | Gaussian Elimination with Backward Substitution |
| 6.2 | Pivoting Strategies — Partial and Scaled Partial |
| 6.3 | Determinant via Row Reduction |
| 6.4 | LU Factorization (Doolittle), without and with pivoting |

**Key idea:** Gaussian elimination transforms $A\mathbf{x} = \mathbf{b}$ to upper-triangular form $U\mathbf{x} = \mathbf{c}$, then solves by back-substitution.  Everything else in the chapter is about doing this *safely* (pivoting) and *reusably* (LU).

In [1]:
import numpy as np
from math import log10, floor

np.set_printoptions(precision=6, suppress=True)

# ── helpers ──────────────────────────────────────────────────────────────────

def fl(x, d=4):
    """Round to d significant digits — simulates d-digit floating-point."""
    if x == 0:
        return 0.0
    magnitude = floor(log10(abs(x)))
    factor = 10 ** (d - 1 - magnitude)
    return round(x * factor) / factor

def fmt_matrix(A, b=None, digits=6):
    """Return a readable string of an augmented (or plain) matrix."""
    n = A.shape[0]
    rows = []
    for i in range(n):
        row = '  [ ' + '  '.join(f'{A[i,j]:>{digits+4}.{digits}g}' for j in range(A.shape[1]))
        if b is not None:
            row += f'  |  {b[i]:>{digits+4}.{digits}g}'
        row += ' ]'
        rows.append(row)
    return '\n'.join(rows)

def back_sub(U, c):
    """Backward substitution: solve Ux = c."""
    n = len(c)
    x = np.zeros(n)
    for i in range(n-1, -1, -1):
        x[i] = (c[i] - U[i, i+1:] @ x[i+1:]) / U[i, i]
    return x

def fwd_sub(L, b):
    """Forward substitution: solve Ly = b (L has 1s on diagonal)."""
    n = len(b)
    y = np.zeros(n)
    for i in range(n):
        y[i] = b[i] - L[i, :i] @ y[:i]
    return y

print('Setup complete.')

Setup complete.


---
## §6.1  Gaussian Elimination — a quick review

**Goal:** reduce the augmented matrix $[A\,|\,\mathbf{b}]$ to upper-triangular form using **row operations**

$$E_i \leftarrow E_i - m_{ij}\,E_j, \quad m_{ij} = \frac{a_{ij}}{a_{jj}}$$

then solve by **backward substitution**:

$$x_n = \frac{a_{n,n+1}}{a_{nn}}, \qquad
x_i = \frac{a_{i,n+1} - \displaystyle\sum_{j=i+1}^{n} a_{ij}\,x_j}{a_{ii}}, \quad i = n-1,\ldots,1.$$

The number $a_{jj}$ used as a divisor is called the **pivot**.

In [2]:
def gaussian_naive(A, b, verbose=True):
    """
    Gaussian elimination WITHOUT any pivoting.
    Returns the solution vector x.
    Raises ZeroDivisionError if a zero pivot is encountered.
    """
    n = len(b)
    Ab = np.hstack([A.astype(float).copy(), b.astype(float).reshape(-1,1).copy()])

    if verbose:
        print('Initial augmented matrix:')
        print(fmt_matrix(Ab[:, :n], Ab[:, n]), '\n')

    for k in range(n - 1):            # elimination steps
        if Ab[k, k] == 0:
            raise ZeroDivisionError(f'Zero pivot at position ({k+1},{k+1}). Use pivoting!')
        for i in range(k + 1, n):
            m = Ab[i, k] / Ab[k, k]
            Ab[i] -= m * Ab[k]
            Ab[i, k] = 0.0
        if verbose:
            print(f'After eliminating column {k+1}:')
            print(fmt_matrix(Ab[:, :n], Ab[:, n]), '\n')

    x = back_sub(Ab[:, :n], Ab[:, n])
    return x


# Quick demo on a well-behaved 3×3 system
# System:  2x1 +  x2 +  x3 =  8
#           x1 + 3x2 +  x3 = 13
#           x1 +  x2 + 4x3 = 14
# True solution: x = [1, 2, 3] (easy to verify by hand)
A_demo = np.array([[2, 1, 1],
                   [1, 3, 1],
                   [1, 1, 4]], dtype=float)
b_demo = np.array([8, 13, 14], dtype=float)

x = gaussian_naive(A_demo, b_demo)
print(f'Solution: x = {x}')
print(f'Verification Ax - b = {A_demo @ x - b_demo}')

Initial augmented matrix:
  [          2           1           1  |           8 ]
  [          1           3           1  |          13 ]
  [          1           1           4  |          14 ] 

After eliminating column 1:
  [          2           1           1  |           8 ]
  [          0         2.5         0.5  |           9 ]
  [          0         0.5         3.5  |          10 ] 

After eliminating column 2:
  [          2           1           1  |           8 ]
  [          0         2.5         0.5  |           9 ]
  [          0           0         3.4  |         8.2 ] 

Solution: x = [1.235294 3.117647 2.411765]
Verification Ax - b = [0. 0. 0.]


---
## §6.2  Why Pivoting is Needed

Naive Gaussian elimination breaks down in two distinct ways:

| Failure mode | Cause | Fix |
|---|---|---|
| **Structural** | Exact zero on the diagonal | Partial pivoting |
| **Numerical** | Near-zero pivot amplifies rounding errors | Partial pivoting |
| **Scale mismatch** | Partial pivoting picks a poor pivot due to differing row scales | *Scaled* partial pivoting |

### 2.1  Case 1 — Exact Zero Pivot (structural failure)

Consider the system

$$\begin{pmatrix} 0 & 1 & 2 \\ 1 & 2 & 3 \\ 2 & 3 & 1 \end{pmatrix}
\begin{pmatrix} x_1 \\ x_2 \\ x_3 \end{pmatrix}
=
\begin{pmatrix} 3 \\ 6 \\ 6 \end{pmatrix}$$

The true solution is $\mathbf{x} = (1,1,1)^T$. But the $(1,1)$ pivot is **zero** — naive elimination crashes immediately.

In [3]:
A_zero = np.array([[0, 1, 2],
                   [1, 2, 3],
                   [2, 3, 1]], dtype=float)
b_zero = np.array([3, 6, 6], dtype=float)

print('Attempting naive Gaussian elimination on a system with a zero pivot...')
try:
    gaussian_naive(A_zero, b_zero, verbose=False)
except ZeroDivisionError as e:
    print(f'ERROR: {e}')

print()
print('NumPy (uses LU with pivoting internally) solves it fine:')
print('  x =', np.linalg.solve(A_zero, b_zero))

Attempting naive Gaussian elimination on a system with a zero pivot...
ERROR: Zero pivot at position (1,1). Use pivoting!

NumPy (uses LU with pivoting internally) solves it fine:
  x = [1. 1. 1.]


### 2.2  Case 2 — Near-Zero Pivot (numerical failure)

A near-zero pivot is not a division-by-zero crash, but it **amplifies rounding errors catastrophically**.

**Example** *(Burden & Faires §6.2, Table 6.3 — simulated 4-digit arithmetic)*

$$\begin{pmatrix} 0.003 & 59.14 \\ 5.291 & -6.130 \end{pmatrix}
\begin{pmatrix} x_1 \\ x_2 \end{pmatrix}
=
\begin{pmatrix} 59.17 \\ 46.78 \end{pmatrix}$$

True solution: $x_1 = 10.00,\; x_2 = 1.000$.

Naive GE picks the $(1,1)$ pivot $= 0.003$. The multiplier $m_{21} = 5.291/0.003 \approx 1764$ then **inflates** row 2 by a factor of ~1764, causing the small coefficient $0.003$ to corrupt the entire computation in low-precision arithmetic.

In [4]:
# ── Simulate 4-digit rounding arithmetic (Burden & Faires §6.2) ──────────────

print('=' * 60)
print('WITHOUT PIVOTING  (4-digit arithmetic simulation)')
print('=' * 60)

# Augmented matrix [A | b]
a11, a12, b1 = 0.003, 59.14, 59.17
a21, a22, b2 = 5.291, -6.130, 46.78

print(f'\nAugmented matrix:\n  [ {a11}  {a12}  |  {b1} ]')
print(f'  [ {a21}  {a22}  |  {b2} ]\n')

# Step 1: compute multiplier (rounded to 4 sig figs)
m = fl(a21 / a11)
print(f'Multiplier m21 = {a21}/{a11} = {a21/a11:.6g}  →  fl(m21) = {m}')

# Step 2: eliminate — apply fl() at every arithmetic operation
a22_new = fl(a22 - fl(m * a12))
b2_new  = fl(b2  - fl(m * b1))
print(f'New a22 = {a22} - {m}×{a12} = {a22 - m*a12:.6g}  →  fl = {a22_new}')
print(f'New b2  = {b2}  - {m}×{b1}  = {b2  - m*b1:.6g}  →  fl = {b2_new}\n')

# Step 3: back-substitute
x2 = fl(b2_new / a22_new)
x1 = fl(fl(b1 - fl(a12 * x2)) / a11)
print(f'x2 = {b2_new}/{a22_new} = {x2}')
print(f'x1 = ({b1} - {a12}×{x2}) / {a11} = {x1}')
print(f'\nResult WITHOUT pivoting:  x1 = {x1}, x2 = {x2}')
print(f'True answer:              x1 = 10.0,  x2 = 1.0')
print(f'Relative error in x1:    {abs(x1 - 10)/10 * 100:.1f}%  ← CATASTROPHIC')

WITHOUT PIVOTING  (4-digit arithmetic simulation)

Augmented matrix:
  [ 0.003  59.14  |  59.17 ]
  [ 5.291  -6.13  |  46.78 ]

Multiplier m21 = 5.291/0.003 = 1763.67  →  fl(m21) = 1764.0
New a22 = -6.13 - 1764.0×59.14 = -104329  →  fl = -104300.0
New b2  = 46.78  - 1764.0×59.17  = -104329  →  fl = -104400.0

x2 = -104400.0/-104300.0 = 1.001
x1 = (59.17 - 59.14×1.001) / 0.003 = -10.0

Result WITHOUT pivoting:  x1 = -10.0, x2 = 1.001
True answer:              x1 = 10.0,  x2 = 1.0
Relative error in x1:    200.0%  ← CATASTROPHIC


In [5]:
print('=' * 60)
print('WITH PARTIAL PIVOTING  (4-digit arithmetic simulation)')
print('=' * 60)

# Row 1: [0.003, 59.14 | 59.17]     scale_1 = 59.14
# Row 2: [5.291, -6.130 | 46.78]    scale_2 = 6.130
# Largest |a_i1|: 5.291 > 0.003  →  SWAP rows

print('\nLargest |element| in column 1:  max(|0.003|, |5.291|) = 5.291  →  swap rows\n')

# After swap
R1 = (5.291, -6.130, 46.78)   # new row 1
R2 = (0.003,  59.14, 59.17)   # new row 2
print(f'After swap:')
print(f'  [ {R1[0]}  {R1[1]}  |  {R1[2]} ]')
print(f'  [ {R2[0]}  {R2[1]}  |  {R2[2]} ]\n')

m = fl(R2[0] / R1[0])
print(f'Multiplier m21 = {R2[0]}/{R1[0]} = {R2[0]/R1[0]:.6g}  →  fl(m21) = {m}')

a22_new = fl(R2[1] - fl(m * R1[1]))
b2_new  = fl(R2[2] - fl(m * R1[2]))
print(f'New a22 = {R2[1]} - {m}×{R1[1]} = {R2[1]-m*R1[1]:.6g}  →  fl = {a22_new}')
print(f'New b2  = {R2[2]} - {m}×{R1[2]} = {R2[2]-m*R1[2]:.6g}  →  fl = {b2_new}\n')

x2 = fl(b2_new / a22_new)
x1 = fl(fl(R1[2] - fl(R1[1] * x2)) / R1[0])
print(f'x2 = {b2_new}/{a22_new} = {x2}')
print(f'x1 = ({R1[2]} - ({R1[1]})×{x2}) / {R1[0]} = {x1}')
print(f'\nResult WITH partial pivoting: x1 = {x1}, x2 = {x2}')
print(f'True answer:                  x1 = 10.0,  x2 = 1.0  ✓')

WITH PARTIAL PIVOTING  (4-digit arithmetic simulation)

Largest |element| in column 1:  max(|0.003|, |5.291|) = 5.291  →  swap rows

After swap:
  [ 5.291  -6.13  |  46.78 ]
  [ 0.003  59.14  |  59.17 ]

Multiplier m21 = 0.003/5.291 = 0.000567001  →  fl(m21) = 0.000567
New a22 = 59.14 - 0.000567×-6.13 = 59.1435  →  fl = 59.14
New b2  = 59.17 - 0.000567×46.78 = 59.1435  →  fl = 59.14

x2 = 59.14/59.14 = 1.0
x1 = (46.78 - (-6.13)×1.0) / 5.291 = 10.0

Result WITH partial pivoting: x1 = 10.0, x2 = 1.0
True answer:                  x1 = 10.0,  x2 = 1.0  ✓


#### Why does the near-zero pivot destroy accuracy?

The multiplier $m_{21} = 5.291 / 0.003 \approx 1764$ amplifies **row 1** when it is subtracted from row 2:

$$a_{22}^{\text{new}} = -6.130 - 1764 \times 59.14 \approx -104{,}300 $$

The original $-6.130$ is completely swamped — it disappears in 4-digit arithmetic.  When we later compute $b_2 - a_{12}x_2$ to find $x_1$, **catastrophic cancellation** wipes out all correct digits.

**Partial pivoting** avoids this: by swapping so the *largest* element in the column becomes the pivot, multipliers satisfy $|m_{ij}| \le 1$, keeping amplification bounded.

### 2.3  Case 3 — When Partial Pivoting Picks the Wrong Row: Scaled Partial Pivoting

Partial pivoting selects the row with the **largest absolute element** in the current column.  But if the rows have very different **scales**, a large element might still be *relatively small* within its row, making it a poor pivot.

**Scaled Partial Pivoting** selects the row maximizing
$$\frac{|a_{ik}|}{s_i}, \qquad s_i = \max_{1 \le j \le n} |a_{ij}|$$
where $s_i$ is computed **once** at the start and kept fixed throughout.

---

**Example** — consider a system where partial pivoting and scaled partial pivoting pick *different* pivot rows:

$$A = \begin{pmatrix} 6 & 60 \\ 4 & 1 \end{pmatrix}, \quad
  \mathbf{b} = \begin{pmatrix} 66 \\ 5 \end{pmatrix}, \quad
  \mathbf{x}^* = \begin{pmatrix} 1 \\ 1 \end{pmatrix}$$

Scale factors:  $s_1 = 60$, $s_2 = 4$.

| Strategy | Row 1 ratio | Row 2 ratio | Pivot row chosen |
|---|---|---|---|
| Partial pivoting | $\|6\| = 6$ | $\|4\| = 4$ | **Row 1** (6 > 4) |
| Scaled partial pivoting | $6/60 = 0.1$ | $4/4 = 1.0$ | **Row 2** (1.0 > 0.1) |

In exact arithmetic both work.  In limited-precision arithmetic, using Row 1 as pivot introduces a multiplier $m_{21} = 4/6$ and then the subtracted term $\tfrac{4}{6} \times 60 = 40$ leaves $a_{22}^{\text{new}} = 1 - 40 = -39$: a dramatic reduction that can cause cancellation when row sizes differ more extremely.

In [6]:
# ── Show the pivot-row selection difference between the two strategies ────────

A_sc = np.array([[6.0, 60.0],
                 [4.0,  1.0]])
b_sc = np.array([66.0, 5.0])

# Scale factors (computed once from original A)
s = np.max(np.abs(A_sc), axis=1)
print(f'Scale factors:  s = {s}  (row-wise max of |a_ij|)')
print()

col = 0  # deciding pivot for column 1
print('Column 1 pivot selection:')
for i in range(2):
    print(f'  Row {i+1}:  |a_{i+1}{col+1}| = {abs(A_sc[i,col]):.4g},  '
          f'|a_{i+1}{col+1}|/s_{i+1} = {abs(A_sc[i,col])/s[i]:.4g}')

pp_row  = np.argmax(np.abs(A_sc[:, col])) + 1
spp_row = np.argmax(np.abs(A_sc[:, col]) / s) + 1
print(f'\nPartial pivoting    →  pivot = Row {pp_row}  (picks |a_i1| = {np.max(np.abs(A_sc[:,col]))})')
print(f'Scaled PP           →  pivot = Row {spp_row}  (picks ratio  = {np.max(np.abs(A_sc[:,col])/s):.4g})')

Scale factors:  s = [60.  4.]  (row-wise max of |a_ij|)

Column 1 pivot selection:
  Row 1:  |a_11| = 6,  |a_11|/s_1 = 0.1
  Row 2:  |a_21| = 4,  |a_21|/s_2 = 1

Partial pivoting    →  pivot = Row 1  (picks |a_i1| = 6.0)
Scaled PP           →  pivot = Row 2  (picks ratio  = 1)


#### A more extreme example — 4-digit arithmetic where partial pivoting gives the wrong answer

$$A = \begin{pmatrix} 2 & 100000 \\ 1 & 1 \end{pmatrix}, \quad
  \mathbf{b} = \begin{pmatrix} 100002 \\ 2 \end{pmatrix}, \quad
  \mathbf{x}^* = \begin{pmatrix} 1 \\ 1 \end{pmatrix}$$

$s_1 = 100000,\; s_2 = 1$.

- **Partial pivoting:** picks Row 1 as pivot ($|2| > |1|$).  
- **Scaled partial pivoting:** picks Row 2 as pivot ($1/1 = 1.0 > 2/100000 = 2\times10^{-5}$).

When we compute $x_1$ with Row 1 as pivot, the key subtraction is
$b_1 - a_{12}\,x_2 = 100002 - 100000 \times 1 = 2$.
In **4-digit arithmetic** $100002 \to 1.000\times10^5$ — the 2 is lost!  $x_1 = 0/2 = 0$. Wrong.

In [7]:
print('=' * 62)
print('PARTIAL PIVOTING — 4-digit arithmetic')
print('=' * 62)

# Original: Row1=[2, 1e5 | 1e5+2], Row2=[1, 1 | 2]
# Partial pivoting: |2| > |1|, no swap needed → Row 1 is pivot
R1 = (fl(2.0),  fl(1e5),   fl(1e5 + 2))
R2 = (fl(1.0),  fl(1.0),   fl(2.0))
print(f'Row1 (pivot): [ {R1[0]}  {R1[1]}  | {R1[2]} ]')
print(f'Row2:         [ {R2[0]}  {R2[1]}  | {R2[2]} ]')
print(f'Note: fl(100002) = {fl(1e5+2)} (the +2 is lost in 4-digit arithmetic!)\n')

m = fl(R2[0] / R1[0])
a22_new = fl(R2[1] - fl(m * R1[1]))
b2_new  = fl(R2[2] - fl(m * R1[2]))
print(f'm21 = {R2[0]}/{R1[0]} = {m}')
print(f'New a22 = {R2[1]} - {m}×{R1[1]} = fl({R2[1] - m*R1[1]:.6g}) = {a22_new}')
print(f'New b2  = {R2[2]} - {m}×{R1[2]} = fl({R2[2] - m*R1[2]:.6g}) = {b2_new}\n')

x2 = fl(b2_new / a22_new)
x1 = fl(fl(R1[2] - fl(R1[1] * x2)) / R1[0])   # R1[2]=fl(1e5+2)=1e5 → cancellation!
print(f'x2 = {b2_new}/{a22_new} = {x2}')
print(f'x1 = (fl(100002) - 1e5×{x2}) / 2 = ({R1[2]} - {fl(R1[1]*x2)}) / {R1[0]} = {x1}')
print(f'\nPartial pivoting result:  x1 = {x1}, x2 = {x2}  ← x1 is WRONG!')

print()
print('=' * 62)
print('SCALED PARTIAL PIVOTING — 4-digit arithmetic')
print('=' * 62)

# Scale factors (computed from ORIGINAL A before any rounding of b)
s1 = 1e5    # max(2, 1e5)
s2 = 1.0    # max(1, 1)
ratio1 = fl(2.0 / s1)   # = 2e-5
ratio2 = fl(1.0 / s2)   # = 1.0
print(f'Scale factors: s1={s1}, s2={s2}')
print(f'Scaled ratios: row1: |2|/{s1}={ratio1}, row2: |1|/{s2}={ratio2}')
print(f'→  SWAP rows (row 2 has larger scaled ratio)\n')

# After swap
P1 = (fl(1.0),  fl(1.0),  fl(2.0))          # was row 2
P2 = (fl(2.0),  fl(1e5),  fl(1e5 + 2))      # was row 1
print(f'After swap:')
print(f'Row1 (pivot): [ {P1[0]}  {P1[1]}  | {P1[2]} ]')
print(f'Row2:         [ {P2[0]}  {P2[1]}  | {P2[2]} ]\n')

m = fl(P2[0] / P1[0])
a22_new = fl(P2[1] - fl(m * P1[1]))
b2_new  = fl(P2[2] - fl(m * P1[2]))
print(f'm21 = {P2[0]}/{P1[0]} = {m}')
print(f'New a22 = {P2[1]} - {m}×{P1[1]} = fl({P2[1]-m*P1[1]:.6g}) = {a22_new}')
print(f'New b2  = {P2[2]} - {m}×{P1[2]} = fl({P2[2]-m*P1[2]:.6g}) = {b2_new}\n')

x2 = fl(b2_new / a22_new)
x1 = fl(fl(P1[2] - fl(P1[1] * x2)) / P1[0])   # P1[2]=2, P1[1]=1 → no cancellation
print(f'x2 = {b2_new}/{a22_new} = {x2}')
print(f'x1 = ({P1[2]} - {P1[1]}×{x2}) / {P1[0]} = {x1}')
print(f'\nScaled partial pivoting: x1 = {x1}, x2 = {x2}  ✓')

PARTIAL PIVOTING — 4-digit arithmetic
Row1 (pivot): [ 2.0  100000.0  | 100000.0 ]
Row2:         [ 1.0  1.0  | 2.0 ]
Note: fl(100002) = 100000.0 (the +2 is lost in 4-digit arithmetic!)

m21 = 1.0/2.0 = 0.5
New a22 = 1.0 - 0.5×100000.0 = fl(-49999) = -50000.0
New b2  = 2.0 - 0.5×100000.0 = fl(-49998) = -50000.0

x2 = -50000.0/-50000.0 = 1.0
x1 = (fl(100002) - 1e5×1.0) / 2 = (100000.0 - 100000.0) / 2.0 = 0.0

Partial pivoting result:  x1 = 0.0, x2 = 1.0  ← x1 is WRONG!

SCALED PARTIAL PIVOTING — 4-digit arithmetic
Scale factors: s1=100000.0, s2=1.0
Scaled ratios: row1: |2|/100000.0=2e-05, row2: |1|/1.0=1.0
→  SWAP rows (row 2 has larger scaled ratio)

After swap:
Row1 (pivot): [ 1.0  1.0  | 2.0 ]
Row2:         [ 2.0  100000.0  | 100000.0 ]

m21 = 2.0/1.0 = 2.0
New a22 = 100000.0 - 2.0×1.0 = fl(99998) = 100000.0
New b2  = 100000.0 - 2.0×2.0 = fl(99996) = 100000.0

x2 = 100000.0/100000.0 = 1.0
x1 = (2.0 - 1.0×1.0) / 1.0 = 1.0

Scaled partial pivoting: x1 = 1.0, x2 = 1.0  ✓


### 2.4  Full Implementations

Below are clean implementations of both pivoting strategies, closely following **Algorithms 6.2** (partial) and **6.3** (scaled partial) from the textbook.

In [8]:
def gaussian_partial(A, b, verbose=False):
    """Gaussian elimination with partial pivoting (Algorithm 6.2)."""
    n = len(b)
    Ab = np.hstack([A.astype(float).copy(), b.astype(float).reshape(-1,1).copy()])
    swaps = 0

    for k in range(n - 1):
        # Find row with max |a_ik| in column k, from row k downward
        p = k + np.argmax(np.abs(Ab[k:, k]))
        if Ab[p, k] == 0:
            raise ValueError('Matrix is singular.')
        if p != k:
            Ab[[k, p]] = Ab[[p, k]]
            swaps += 1
            if verbose:
                print(f'  Step {k+1}: swap rows {k+1} ↔ {p+1}')

        for i in range(k + 1, n):
            m = Ab[i, k] / Ab[k, k]
            Ab[i] -= m * Ab[k]
            Ab[i, k] = 0.0

    return back_sub(Ab[:, :n], Ab[:, n]), swaps


def gaussian_scaled(A, b, verbose=False):
    """Gaussian elimination with scaled partial pivoting (Algorithm 6.3)."""
    n = len(b)
    Ab = np.hstack([A.astype(float).copy(), b.astype(float).reshape(-1,1).copy()])
    # Scale factors computed ONCE from the original matrix
    s = np.max(np.abs(A), axis=1).astype(float)
    if np.any(s == 0):
        raise ValueError('Zero row in matrix — singular.')
    swaps = 0

    for k in range(n - 1):
        # Find row with max |a_ik| / s_i, from row k downward
        ratios = np.abs(Ab[k:, k]) / s[k:]
        p = k + np.argmax(ratios)
        if Ab[p, k] == 0:
            raise ValueError('Matrix is singular.')
        if p != k:
            Ab[[k, p]] = Ab[[p, k]]
            s[[k, p]]  = s[[p, k]]   # swap scale factors alongside rows
            swaps += 1
            if verbose:
                print(f'  Step {k+1}: swap rows {k+1} ↔ {p+1}  '
                      f'(ratios: {ratios})')

        for i in range(k + 1, n):
            m = Ab[i, k] / Ab[k, k]
            Ab[i] -= m * Ab[k]
            Ab[i, k] = 0.0

    return back_sub(Ab[:, :n], Ab[:, n]), swaps


# ── Verify on the Burden & Faires near-zero-pivot example ────────────────────
A_bfex = np.array([[0.003,  59.14],
                   [5.291, -6.130]], dtype=float)
b_bfex = np.array([59.17, 46.78], dtype=float)

x_pp, _  = gaussian_partial(A_bfex, b_bfex)
x_sp, _  = gaussian_scaled(A_bfex,  b_bfex)
x_np     = np.linalg.solve(A_bfex, b_bfex)

print('Burden & Faires near-zero-pivot example:')
print(f'  Partial pivot result:   x1 = {x_pp[0]:.6f},  x2 = {x_pp[1]:.6f}')
print(f'  Scaled pivot result:    x1 = {x_sp[0]:.6f},  x2 = {x_sp[1]:.6f}')
print(f'  NumPy (reference):      x1 = {x_np[0]:.6f},  x2 = {x_np[1]:.6f}')
print(f'  True solution:          x1 = 10.000000,  x2 = 1.000000')

Burden & Faires near-zero-pivot example:
  Partial pivot result:   x1 = 10.000000,  x2 = 1.000000
  Scaled pivot result:    x1 = 10.000000,  x2 = 1.000000
  NumPy (reference):      x1 = 10.000000,  x2 = 1.000000
  True solution:          x1 = 10.000000,  x2 = 1.000000


---
## §6.3  Determinant via Gaussian Elimination

Three facts connect row operations to the determinant:

| Row operation | Effect on $\det$ |
|---|---|
| $E_i \leftarrow E_i - m\,E_k$ | **No change** |
| $E_i \leftrightarrow E_j$ (row swap) | **Multiply by $-1$** |
| $E_i \leftarrow c\,E_i$ | **Multiply by $c$** |

After reducing $A$ to upper-triangular $U$ via partial pivoting with $k$ swaps:

$$\det(A) = (-1)^k \prod_{i=1}^{n} u_{ii}$$

**This is exactly what `numpy.linalg.det` does internally (via LU factorization).**

In [9]:
def determinant_ge(A, verbose=True):
    """
    Compute det(A) using Gaussian elimination with partial pivoting.
    Returns the determinant.
    """
    n = A.shape[0]
    M = A.astype(float).copy()
    swaps = 0

    for k in range(n - 1):
        p = k + np.argmax(np.abs(M[k:, k]))
        if M[p, k] == 0:
            if verbose:
                print('Zero pivot encountered → det = 0 (singular matrix)')
            return 0.0
        if p != k:
            M[[k, p]] = M[[p, k]]
            swaps += 1
            if verbose:
                print(f'  Swap rows {k+1} ↔ {p+1}')

        for i in range(k + 1, n):
            m = M[i, k] / M[k, k]
            M[i] -= m * M[k]
            M[i, k] = 0.0

    diag = np.diag(M)
    det  = ((-1) ** swaps) * np.prod(diag)

    if verbose:
        print(f'\nUpper-triangular diagonal: {diag}')
        print(f'Number of row swaps:        {swaps}')
        print(f'det = (-1)^{swaps} × {" × ".join(f"{d:.4g}" for d in diag)}')
        print(f'    = {det:.6g}')

    return det

In [10]:
# ── Example 1: 3×3 matrix with known determinant ─────────────────────────────
#
# A = | 1  2  3 |   Expanding along row 1:
#     | 4  5  6 |   det = 1(5·9−6·8) − 2(4·9−6·7) + 3(4·8−5·7)
#     | 7  8  9 |        = 1(−3) − 2(−6) + 3(−3) = −3+12−9 = 0
#
# This matrix IS singular (rows are in arithmetic progression).

A1 = np.array([[1, 2, 3],
               [4, 5, 6],
               [7, 8, 9]], dtype=float)

print('Example 1 — singular 3×3 (rows in AP):')
print(f'A =\n{A1}\n')
d = determinant_ge(A1)
print(f'\nNumPy check: {np.linalg.det(A1):.6g}')

Example 1 — singular 3×3 (rows in AP):
A =
[[1. 2. 3.]
 [4. 5. 6.]
 [7. 8. 9.]]

  Swap rows 1 ↔ 3
  Swap rows 2 ↔ 3

Upper-triangular diagonal: [7.       0.857143 0.      ]
Number of row swaps:        2
det = (-1)^2 × 7 × 0.8571 × 1.11e-16
    = 6.66134e-16

NumPy check: 0


In [11]:
# ── Example 2: 4×4 with nonzero determinant ───────────────────────────────────
#
# From Burden & Faires, Exercise 6.1 / 6.3 style:
#
# A = | 1   2  -1   1 |
#     | 2   5   0   2 |
#     | 3   7   2   5 |
#     |-1  -2   3  -1 |

A2 = np.array([[ 1,  2, -1,  1],
               [ 2,  5,  0,  2],
               [ 3,  7,  2,  5],
               [-1, -2,  3, -1]], dtype=float)

print('Example 2 — general 4×4:')
print(f'A =\n{A2}\n')
d = determinant_ge(A2)
print(f'\nNumPy check: {np.linalg.det(A2):.6g}')

Example 2 — general 4×4:
A =
[[ 1.  2. -1.  1.]
 [ 2.  5.  0.  2.]
 [ 3.  7.  2.  5.]
 [-1. -2.  3. -1.]]

  Swap rows 1 ↔ 3
  Swap rows 3 ↔ 4

Upper-triangular diagonal: [ 3.        0.333333  5.       -0.8     ]
Number of row swaps:        2
det = (-1)^2 × 3 × 0.3333 × 5 × -0.8
    = -4

NumPy check: -4


---
## §6.4  LU Factorization

**Key idea:** instead of discarding the elimination steps, *save* the multipliers $m_{ij}$ as the lower-triangular factor $L$.  Then for any new right-hand side $\mathbf{b}$:

$$A = LU \implies A\mathbf{x}=\mathbf{b} \;\Longleftrightarrow\; \begin{cases} L\mathbf{y} = \mathbf{b} & \text{(forward substitution)} \\ U\mathbf{x} = \mathbf{y} & \text{(backward substitution)} \end{cases}$$

**Doolittle's method** sets $l_{ii} = 1$ (unit lower-triangular $L$).

**Crout's method** sets $u_{ii} = 1$.

We follow **Doolittle** (used in Burden & Faires Algorithm 6.4).

### Doolittle Formulas

For $k = 1, 2, \ldots, n$:

$$u_{kj} = a_{kj} - \sum_{s=1}^{k-1} l_{ks}\,u_{sj}, \qquad j = k, k+1, \ldots, n$$

$$l_{ik} = \frac{1}{u_{kk}}\!\left(a_{ik} - \sum_{s=1}^{k-1} l_{is}\,u_{sk}\right), \qquad i = k+1, \ldots, n$$

In [12]:
def lu_doolittle(A, verbose=True):
    """
    Doolittle LU factorization WITHOUT pivoting.
    Returns L, U such that A = L @ U.
    L has 1s on its diagonal.
    """
    n = A.shape[0]
    L = np.eye(n)
    U = np.zeros((n, n))

    for k in range(n):
        # ── Row k of U ────────────────────────────────────────────────────────
        for j in range(k, n):
            U[k, j] = A[k, j] - L[k, :k] @ U[:k, j]

        if U[k, k] == 0:
            raise ValueError(f'Zero pivot u_{k+1}{k+1} — use pivoting.')

        # ── Column k of L ─────────────────────────────────────────────────────
        for i in range(k + 1, n):
            L[i, k] = (A[i, k] - L[i, :k] @ U[:k, k]) / U[k, k]

        if verbose:
            print(f'After step k={k+1}:')
            print(f'  U[{k+1}, :] = {U[k, :k+1+(n-k-1)]}')
            if k < n-1:
                print(f'  L[:, {k+1}] = {L[k+1:, k]}')
            print()

    return L, U

In [13]:
# ── Example (Burden & Faires §6.4, Example 1) ─────────────────────────────────
#
# A = |  1   1   0   3 |
#     |  2   1  -1   1 |
#     |  3  -1  -1   2 |
#     | -1   2   3  -1 |
#
# b = | 8 |    true solution: x = (1, 2, -1, 1)
#     | 7 |    verify: 1+2+0+3=6? No, let's use b = Ax* below.
#     | 14|
#     |-7 |

A_lu = np.array([[ 1,  1,  0,  3],
                 [ 2,  1, -1,  1],
                 [ 3, -1, -1,  2],
                 [-1,  2,  3, -1]], dtype=float)

x_true = np.array([1.0, 2.0, -1.0, 1.0])
b_lu   = A_lu @ x_true
print(f'System: A @ x = b,   b = {b_lu},   true x = {x_true}\n')
print('=' * 50)
print('Doolittle LU Factorization (step by step):')
print('=' * 50)

L, U = lu_doolittle(A_lu, verbose=True)

print('L =')
print(L)
print('\nU =')
print(U)
print(f'\nVerification: max|LU - A| = {np.max(np.abs(L @ U - A_lu)):.2e}')

System: A @ x = b,   b = [ 6.  6.  4. -1.],   true x = [ 1.  2. -1.  1.]

Doolittle LU Factorization (step by step):
After step k=1:
  U[1, :] = [1. 1. 0. 3.]
  L[:, 1] = [ 2.  3. -1.]

After step k=2:
  U[2, :] = [ 0. -1. -1. -5.]
  L[:, 2] = [ 4. -3.]

After step k=3:
  U[3, :] = [ 0.  0.  3. 13.]
  L[:, 3] = [0.]

After step k=4:
  U[4, :] = [  0.   0.   0. -13.]

L =
[[ 1.  0.  0.  0.]
 [ 2.  1.  0.  0.]
 [ 3.  4.  1.  0.]
 [-1. -3.  0.  1.]]

U =
[[  1.   1.   0.   3.]
 [  0.  -1.  -1.  -5.]
 [  0.   0.   3.  13.]
 [  0.   0.   0. -13.]]

Verification: max|LU - A| = 0.00e+00


In [14]:
# ── Solve Ax = b via the LU factors ──────────────────────────────────────────
print('Solving Ax = b using L and U:')
print()
print('Step 1 — Forward substitution: Ly = b')
y = fwd_sub(L, b_lu)
print(f'  y = {y}')
print(f'  Verification: max|Ly - b| = {np.max(np.abs(L @ y - b_lu)):.2e}')

print()
print('Step 2 — Backward substitution: Ux = y')
x = back_sub(U, y)
print(f'  x = {x}')
print(f'  True solution: {x_true}')
print(f'  Verification: max|Ax - b| = {np.max(np.abs(A_lu @ x - b_lu)):.2e}')

Solving Ax = b using L and U:

Step 1 — Forward substitution: Ly = b
  y = [  6.  -6.  10. -13.]
  Verification: max|Ly - b| = 0.00e+00

Step 2 — Backward substitution: Ux = y
  x = [ 1.  2. -1.  1.]
  True solution: [ 1.  2. -1.  1.]
  Verification: max|Ax - b| = 0.00e+00


### Why LU is better than repeated Gaussian elimination

If we need to solve $A\mathbf{x} = \mathbf{b}_1, A\mathbf{x} = \mathbf{b}_2, \ldots, A\mathbf{x} = \mathbf{b}_m$:

- **Repeated GE:** $m \times O(n^3)$ operations.
- **LU factorization once:** $O(n^3)$ for the factorization + $m \times O(n^2)$ for forward/back substitutions.

For $m \gg 1$ this is a dramatic saving.

In [15]:
# ── Solve for multiple right-hand sides using ONE LU factorization ────────────
B = np.array([[8, 4, -1],
              [7, 3,  0],
              [14,9,  2],
              [-7,1,  5]], dtype=float)

print('Solving Ax = b for 3 different right-hand sides using one LU:')
for j in range(B.shape[1]):
    y_j = fwd_sub(L, B[:, j])
    x_j = back_sub(U, y_j)
    err = np.max(np.abs(A_lu @ x_j - B[:, j]))
    print(f'  b_{j+1} = {B[:,j]}  →  x_{j+1} = {np.round(x_j,4)}  (residual {err:.1e})')

Solving Ax = b for 3 different right-hand sides using one LU:
  b_1 = [ 8.  7. 14. -7.]  →  x_1 = [ 3. -1.  0.  2.]  (residual 0.0e+00)
  b_2 = [4. 3. 9. 1.]  →  x_2 = [ 2.8718 -1.1795  2.3333  0.7692]  (residual 8.9e-16)
  b_3 = [-1.  0.  2.  5.]  →  x_3 = [ 1.7949 -0.4872  2.3333 -0.7692]  (residual 8.9e-16)


---
### LU Factorization WITH Pivoting  ($PA = LU$)

When a zero (or near-zero) pivot is encountered during Doolittle, we again need pivoting.  Incorporating row swaps gives:

$$PA = LU$$

where $P$ is a **permutation matrix** (rows of the identity, reordered).

**To solve $A\mathbf{x} = \mathbf{b}$:**

$$PA\mathbf{x} = P\mathbf{b} \implies LU\mathbf{x} = P\mathbf{b}$$

1. Forward-substitute: $L\mathbf{y} = P\mathbf{b}$
2. Back-substitute:    $U\mathbf{x} = \mathbf{y}$

In [16]:
def lu_pivot(A, verbose=True):
    """
    LU factorization with partial pivoting.  Returns P, L, U such that P @ A = L @ U.
    P is a permutation matrix.  L is unit lower-triangular.
    """
    n = A.shape[0]
    U = A.astype(float).copy()
    L = np.eye(n)
    P = np.eye(n)

    for k in range(n - 1):
        # Partial pivot
        p = k + np.argmax(np.abs(U[k:, k]))
        if U[p, k] == 0:
            raise ValueError('Singular matrix.')

        if p != k:
            U[[k, p]]      = U[[p, k]]
            P[[k, p]]      = P[[p, k]]
            # Swap already-computed columns of L (below diagonal, columns 0..k-1)
            if k > 0:
                L[[k, p], :k] = L[[p, k], :k]
            if verbose:
                print(f'  Step k={k+1}: swap rows {k+1} ↔ {p+1}')

        for i in range(k + 1, n):
            L[i, k] = U[i, k] / U[k, k]
            U[i]   -= L[i, k] * U[k]
            U[i, k] = 0.0

    return P, L, U


def lu_solve(P, L, U, b):
    """Solve Ax=b given P,L,U from lu_pivot: PA=LU → LUx = Pb."""
    Pb = P @ b
    y  = fwd_sub(L, Pb)
    x  = back_sub(U, y)
    return x

In [17]:
# ── Example: matrix that REQUIRES pivoting (zero diagonal after one step) ─────
#
# A = | 0   2   1 |     Without pivoting: zero pivot at (1,1)
#     | 1   2   3 |     true solution: x = (1, 1, 1)
#     | 2   3   1 |

A_piv = np.array([[0, 2, 1],
                  [1, 2, 3],
                  [2, 3, 1]], dtype=float)
x_true_piv = np.array([1.0, 1.0, 1.0])
b_piv = A_piv @ x_true_piv
print(f'System: b = {b_piv},   true x = {x_true_piv}\n')

print('LU with partial pivoting (PA = LU):')
P, L, U = lu_pivot(A_piv, verbose=True)

print(f'\nP =\n{P.astype(int)}')
print(f'\nL =\n{L}')
print(f'\nU =\n{U}')
print(f'\nVerification: max|PA - LU| = {np.max(np.abs(P @ A_piv - L @ U)):.2e}')

System: b = [3. 6. 6.],   true x = [1. 1. 1.]

LU with partial pivoting (PA = LU):
  Step k=1: swap rows 1 ↔ 3
  Step k=2: swap rows 2 ↔ 3

P =
[[0 0 1]
 [1 0 0]
 [0 1 0]]

L =
[[1.   0.   0.  ]
 [0.   1.   0.  ]
 [0.5  0.25 1.  ]]

U =
[[2.   3.   1.  ]
 [0.   2.   1.  ]
 [0.   0.   2.25]]

Verification: max|PA - LU| = 0.00e+00


In [18]:
print('Solving Ax = b using PA = LU:')
print()
Pb = P @ b_piv
print(f'1. Permute b:  Pb = {Pb}')

y = fwd_sub(L, Pb)
print(f'2. Forward sub Ly = Pb:  y = {y}')

x = back_sub(U, y)
print(f'3. Back sub Ux = y:      x = {x}')

print(f'\nTrue solution:           x = {x_true_piv}')
print(f'Residual max|Ax-b| = {np.max(np.abs(A_piv @ x - b_piv)):.2e}')

Solving Ax = b using PA = LU:

1. Permute b:  Pb = [6. 3. 6.]
2. Forward sub Ly = Pb:  y = [6.   3.   2.25]
3. Back sub Ux = y:      x = [1. 1. 1.]

True solution:           x = [1. 1. 1.]
Residual max|Ax-b| = 0.00e+00


In [19]:
# ── Larger example: 4×4 with pivoting, compare to scipy ──────────────────────
from scipy.linalg import lu as scipy_lu

A_4x4 = np.array([[ 2,  1,  1,  0],
                  [ 4,  3,  3,  1],
                  [ 8,  7,  9,  5],
                  [ 6,  7,  9,  8]], dtype=float)

x_true_4 = np.array([1.0, 1.0, 1.0, 1.0])
b_4      = A_4x4 @ x_true_4

# Our implementation
P_our, L_our, U_our = lu_pivot(A_4x4, verbose=False)
x_our = lu_solve(P_our, L_our, U_our, b_4)

# SciPy reference (note: scipy returns P, L, U where A = P @ L @ U)
P_sci, L_sci, U_sci = scipy_lu(A_4x4)

print('4×4 LU with pivoting:')
print(f'\nOur  P =\n{P_our.astype(int)}')
print(f'\nOur  L =\n{L_our}')
print(f'\nOur  U =\n{U_our}')
print(f'\nPA = LU check: max|PA - LU| = {np.max(np.abs(P_our @ A_4x4 - L_our @ U_our)):.2e}')
print(f'\nSolution:  x = {x_our}')
print(f'True:      x = {x_true_4}')
print(f'Residual:      {np.max(np.abs(A_4x4 @ x_our - b_4)):.2e}')

4×4 LU with pivoting:

Our  P =
[[0 0 1 0]
 [0 0 0 1]
 [0 1 0 0]
 [1 0 0 0]]

Our  L =
[[ 1.        0.        0.        0.      ]
 [ 0.75      1.        0.        0.      ]
 [ 0.5      -0.285714  1.        0.      ]
 [ 0.25     -0.428571  0.333333  1.      ]]

Our  U =
[[ 8.        7.        9.        5.      ]
 [ 0.        1.75      2.25      4.25    ]
 [ 0.        0.       -0.857143 -0.285714]
 [ 0.        0.        0.        0.666667]]

PA = LU check: max|PA - LU| = 1.11e-16

Solution:  x = [1. 1. 1. 1.]
True:      x = [1. 1. 1. 1.]
Residual:      0.00e+00


---
## Summary — Comparison of All Methods

| Method | Handles zero pivot | Numerically stable | Reusable for multiple RHS | Notes |
|---|---|---|---|---|
| Naive GE | ✗ | ✗ | ✗ | textbook baseline only |
| GE + Partial Pivot | ✓ | ✓ (mostly) | ✗ | standard choice |
| GE + Scaled Partial Pivot | ✓ | ✓✓ | ✗ | better for ill-scaled systems |
| LU (Doolittle) | ✗ | ✗ | ✓ | fast for multiple RHS |
| LU + Pivoting ($PA=LU$) | ✓ | ✓ | ✓ | **production standard** |

In [20]:
# ── Side-by-side comparison on the Burden & Faires near-zero example ─────────
print('Comparison on 0.003/59.14 system (true: x1=10, x2=1)\n')

results = {}
results['Naive GE (exact float64)'] = gaussian_naive(A_bfex, b_bfex, verbose=False)
results['Partial pivoting'],    _   = gaussian_partial(A_bfex, b_bfex)
results['Scaled partial pivot'], _  = gaussian_scaled(A_bfex,  b_bfex)
P_c, L_c, U_c = lu_pivot(A_bfex, verbose=False)
results['LU + pivot (PA=LU)']       = lu_solve(P_c, L_c, U_c, b_bfex)
results['NumPy reference']          = np.linalg.solve(A_bfex, b_bfex)

print(f'{"Method":<30}  {"x1":>12}  {"x2":>12}  {"err x1":>12}')
print('-' * 72)
for name, x in results.items():
    err = abs(x[0] - 10.0)
    print(f'{name:<30}  {x[0]:>12.6f}  {x[1]:>12.6f}  {err:>12.2e}')

Comparison on 0.003/59.14 system (true: x1=10, x2=1)

Method                                    x1            x2        err x1
------------------------------------------------------------------------
Naive GE (exact float64)           10.000000      1.000000      3.78e-13
Partial pivoting                   10.000000      1.000000      0.00e+00
Scaled partial pivot               10.000000      1.000000      0.00e+00
LU + pivot (PA=LU)                 10.000000      1.000000      0.00e+00
NumPy reference                    10.000000      1.000000      0.00e+00


---
## Practice Exercises

**Exercise 1** *(Zero pivot / partial pivoting)*  
Solve the following system using partial pivoting, showing every row swap and elimination step:
$$\begin{pmatrix} 0 & 1 & 1 \\ 2 & 3 & 5 \\ 1 & 2 & 4 \end{pmatrix}\mathbf{x} = \begin{pmatrix} 2 \\ 10 \\ 7 \end{pmatrix}$$

**Exercise 2** *(Scaled partial pivoting selection)*  
For column 1 of the matrix
$$A = \begin{pmatrix} 30 & 591400 \\ 5.291 & -6.130 \end{pmatrix}$$
compute the scale factors $s_i$ and the scaled ratios $|a_{i1}|/s_i$.  
Which row does partial pivoting select?  Which does scaled partial pivoting select?  
*(This is Table 6.4 from Burden & Faires.)*

**Exercise 3** *(Determinant)*  
Use Gaussian elimination to find $\det(A)$ for
$$A = \begin{pmatrix} 2 & -1 & 0 \\ -1 & 2 & -1 \\ 0 & -1 & 2 \end{pmatrix}$$
*(This is the tridiagonal matrix from finite differences; $\det = 4$.)*

**Exercise 4** *(LU factorization by hand)*  
Compute the Doolittle factorization $A = LU$ for
$$A = \begin{pmatrix} 2 & 1 & 1 \\ 4 & 3 & 3 \\ 8 & 7 & 9 \end{pmatrix}$$
Then solve $A\mathbf{x} = (1, 1, 1)^T$.

**Exercise 5** *(PA = LU)*  
Apply LU factorization with partial pivoting to the matrix in Exercise 4 appended with a zero in position $(1,1)$:
$$A' = \begin{pmatrix} 0 & 1 & 1 \\ 4 & 3 & 3 \\ 8 & 7 & 9 \end{pmatrix}$$
Find $P$, $L$, $U$ and verify $PA' = LU$.

In [21]:
# ── Exercise solutions (run to check your work) ───────────────────────────────

print('Exercise 1:')
A_ex1 = np.array([[0, 1, 1], [2, 3, 5], [1, 2, 4]], dtype=float)
b_ex1 = np.array([2, 10, 7], dtype=float)
x1, _ = gaussian_partial(A_ex1, b_ex1, verbose=True)
print(f'Solution: x = {x1}  (verify Ax-b: {A_ex1@x1 - b_ex1})')

print('\nExercise 2:')
A_ex2 = np.array([[30.0, 591400.0], [5.291, -6.130]])
s_ex2 = np.max(np.abs(A_ex2), axis=1)
print(f'Scale factors s = {s_ex2}')
print(f'|a_i1|:        {np.abs(A_ex2[:,0])}')
print(f'Partial pivot picks row:  {np.argmax(np.abs(A_ex2[:,0]))+1} (largest |a_i1|)')
print(f'Scaled ratios: {np.abs(A_ex2[:,0])/s_ex2}')
print(f'Scaled pivot picks row:   {np.argmax(np.abs(A_ex2[:,0])/s_ex2)+1}')

print('\nExercise 3:')
A_ex3 = np.array([[2,-1,0],[-1,2,-1],[0,-1,2]], dtype=float)
d3 = determinant_ge(A_ex3, verbose=True)
print(f'det = {d3} (expected: 4)')

print('\nExercise 4:')
A_ex4 = np.array([[2,1,1],[4,3,3],[8,7,9]], dtype=float)
L4, U4 = lu_doolittle(A_ex4, verbose=False)
print(f'L =\n{L4}\nU =\n{U4}')
y4 = fwd_sub(L4, np.ones(3))
x4 = back_sub(U4, y4)
print(f'Solution x = {x4}')

print('\nExercise 5:')
A_ex5 = np.array([[0,1,1],[4,3,3],[8,7,9]], dtype=float)
P5, L5, U5 = lu_pivot(A_ex5, verbose=True)
print(f'P =\n{P5.astype(int)}\nL =\n{L5}\nU =\n{U5}')
print(f'PA - LU max error: {np.max(np.abs(P5 @ A_ex5 - L5 @ U5)):.2e}')

Exercise 1:
  Step 1: swap rows 1 ↔ 2
Solution: x = [1. 1. 1.]  (verify Ax-b: [0. 0. 0.])

Exercise 2:
Scale factors s = [591400.        6.13]
|a_i1|:        [30.     5.291]
Partial pivot picks row:  1 (largest |a_i1|)
Scaled ratios: [0.000051 0.863132]
Scaled pivot picks row:   2

Exercise 3:

Upper-triangular diagonal: [2.       1.5      1.333333]
Number of row swaps:        0
det = (-1)^0 × 2 × 1.5 × 1.333
    = 4
det = 4.0 (expected: 4)

Exercise 4:
L =
[[1. 0. 0.]
 [2. 1. 0.]
 [4. 3. 1.]]
U =
[[2. 1. 1.]
 [0. 1. 1.]
 [0. 0. 2.]]
Solution x = [ 1. -1.  0.]

Exercise 5:
  Step k=1: swap rows 1 ↔ 3
  Step k=2: swap rows 2 ↔ 3
P =
[[0 0 1]
 [1 0 0]
 [0 1 0]]
L =
[[ 1.   0.   0. ]
 [ 0.   1.   0. ]
 [ 0.5 -0.5  1. ]]
U =
[[ 8.  7.  9.]
 [ 0.  1.  1.]
 [ 0.  0. -1.]]
PA - LU max error: 0.00e+00
